In [1]:
import pandas as pd

df = pd.read_csv("combined/train_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7977, 244)


,rssi_RX1,rssi_RX2,RX1_mean_0,RX1_std_0,RX2_mean_0,RX2_std_0,RX1_mean_1,RX1_std_1,RX2_mean_1,RX2_std_1,...,diff_40,diff_41,diff_42,diff_43,diff_44,diff_45,diff_46,diff_47,person_count,scenario_id
0,-48.4,-51.3,-1.4,2.835489,-0.1,3.047950,0.4,2.537716,-1.3,1.676305,...,-9.0,-8.8,1.7,-8.4,-7.7,0.6,-6.9,-1.7,0,No_person
1,-48.3,-51.5,-1.0,2.683282,-0.8,2.749545,0.8,2.749545,-0.9,2.211334,...,-4.7,-4.6,-0.3,-4.3,-4.3,-1.0,-4.1,-1.2,0,No_person
2,-48.3,-51.3,-1.2,2.271563,-0.8,2.357965,0.6,3.352611,-0.1,2.624881,...,-0.9,-1.4,2.5,-1.8,-2.8,2.7,-3.2,-1.0,0,No_person
3,-48.5,-51.5,-1.4,2.154066,-0.4,2.009975,1.2,3.187475,-0.4,2.870540,...,-4.0,-4.6,2.0,-5.7,-7.1,1.3,-8.1,-2.3,0,No_person
4,-48.6,-51.6,-1.4,1.959592,0.3,1.791647,0.4,3.527038,-0.5,2.765863,...,-0.9,-1.7,2.5,-3.1,-4.7,1.5,-5.8,-1.6,0,No_person


In [2]:
y = df["person_count"]
X = df.drop(columns=["person_count", "scenario_id"])
groups = df["scenario_id"]

print("Overall class distribution:")
print(y.value_counts())

Overall class distribution:
person_count
2    3051
1    2875
3    1033
0    1018
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Val shape  :", X_val.shape)

print("\nTrain distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation distribution:")
print(y_val.value_counts().sort_index())

Train shape: (6381, 242)
Val shape  : (1596, 242)

Train distribution:
person_count
0     814
1    2300
2    2441
3     826
Name: count, dtype: int64

Validation distribution:
person_count
0    204
1    575
2    610
3    207
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

print("✅ Scaling done")

✅ Scaling done


In [5]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("✅ Model trained")

✅ Model trained


In [6]:
from xgboost import XGBClassifier

model2 = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)
model2.fit(X_train, y_train)
print("✅ Model trained")

✅ Model trained


In [7]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

val_preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, val_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))

print("\nClassification Report:")
print(classification_report(y_val, val_preds, zero_division=0))

mae = np.mean(np.abs(y_val.values - val_preds))
print("\nMean Absolute Error:", mae)

tol_acc = np.mean(np.abs(y_val.values - val_preds) <= 1)
print("Accuracy within ±1 person:", tol_acc)

Accuracy: 0.968671679197995

Confusion Matrix:
[[200   1   3   0]
 [  6 561   8   0]
 [  8  13 580   9]
 [  0   0   2 205]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.96       204
           1       0.98      0.98      0.98       575
           2       0.98      0.95      0.96       610
           3       0.96      0.99      0.97       207

    accuracy                           0.97      1596
   macro avg       0.96      0.97      0.97      1596
weighted avg       0.97      0.97      0.97      1596


Mean Absolute Error: 0.03822055137844611
Accuracy within ±1 person: 0.9931077694235589


In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

val_preds = model2.predict(X_val)

print("Accuracy:", accuracy_score(y_val, val_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))

print("\nClassification Report:")
print(classification_report(y_val, val_preds, zero_division=0))

mae = np.mean(np.abs(y_val.values - val_preds))
print("\nMean Absolute Error:", mae)

tol_acc = np.mean(np.abs(y_val.values - val_preds) <= 1)
print("Accuracy within ±1 person:", tol_acc)

Accuracy: 0.981829573934837

Confusion Matrix:
[[201   1   2   0]
 [  4 565   6   0]
 [  3   9 594   4]
 [  0   0   0 207]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       204
           1       0.98      0.98      0.98       575
           2       0.99      0.97      0.98       610
           3       0.98      1.00      0.99       207

    accuracy                           0.98      1596
   macro avg       0.98      0.99      0.98      1596
weighted avg       0.98      0.98      0.98      1596


Mean Absolute Error: 0.021303258145363407
Accuracy within ±1 person: 0.9968671679197995


In [9]:
import joblib

joblib.dump(model, "rf_person_count_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

print("✅ Model, scaler, and features saved")

✅ Model, scaler, and features saved


In [10]:
#import json
#importance = pd.Series(model.feature_importances_, index=X.columns)
#importance = importance.sort_values(ascending=False)

#top_features = importance.head(50).index

#selected_idx = set()

#for feat in top_features:
#    parts = feat.split("_")
#    if parts[-1].isdigit():
#        selected_idx.add(int(parts[-1]))

#selected_idx = sorted(list(selected_idx))

#print("Selected indices:", selected_idx)

## Save indices
#with open("selected_idx.json", "w") as f:
#    json.dump(selected_idx, f)